# Kateto — ORPO v7 (305 pares, 2xT4)

Corre ORPO sobre el SFT v7 con instrumentación `[orpo-diag]`: cada paso emite una línea `[orpo-diag step=...]` con la finitud de cada término de la loss.
La base es `kateto-v7-sft`; el código de entrenamiento viene vendoreado de `kateto-rwkv-peft` (RWKV-PEFT está en `.gitignore` y el clone de GitHub NO lo trae).


In [ ]:
# CELL 1 — setup
import os, subprocess, sys, shutil, importlib.util
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],capture_output=True,text=True).stdout)
subprocess.run([sys.executable,'-m','pip','install','-q',
                'lightning','datasets','safetensors','transformers','jsonlines',
                'rwkv','peft==0.20.0','einops','rwkv-fla'],check=True)
# Kaggle trae torchao 0.10.0 y peft 0.20.0 lo rechaza en dispatch_torchao (ImportError).
# Sin torchao instalado, peft.is_torchao_available() devuelve False y el dispatcher sigue.
subprocess.run([sys.executable,'-m','pip','uninstall','-y','-q','torchao'],check=False)
import peft as _peft, importlib.util as _ilu
print('[env] peft', _peft.__version__, 'torchao_instalado', _ilu.find_spec('torchao') is not None)
import importlib
_faltan = [m for m in ('rwkv','peft','einops','jsonlines','lightning','datasets','transformers','rwkvfla')
           if importlib.util.find_spec(m) is None]
assert not _faltan, f'faltan modulos: {_faltan}'
print('[ok] imports listos:', 'rwkv peft einops jsonlines lightning datasets transformers')
shutil.copytree('/kaggle/input/kateto-code-orpo', '/kaggle/working/kateto', dirs_exist_ok=True)
os.chdir('/kaggle/working/kateto')

if os.path.isdir('/kaggle/working/kateto/.git'):
    print(subprocess.run(['git','log','--oneline','-3'],capture_output=True,text=True).stdout)
else:
    print('[ok] sin .git (arbol vendoreado)')

assert os.path.exists('RWKV-PEFT/train.py'), 'no existe RWKV-PEFT/train.py'
assert os.path.exists('RWKV-PEFT/rwkvt/lightning_train/light_rwkv.py'), 'no existe RWKV-PEFT/rwkvt/lightning_train/light_rwkv.py'
n = open('RWKV-PEFT/rwkvt/lightning_train/light_rwkv.py').read().count('orpo-diag')
assert n > 0, 'RWKV-PEFT/rwkvt/lightning_train/light_rwkv.py no contiene orpo-diag'
assert os.path.exists('scripts/smoke-orpo-1.5b-local.sh'), 'no existe scripts/smoke-orpo-1.5b-local.sh'
assert 'SMOKE_CLIP' in open('scripts/smoke-orpo-1.5b-local.sh').read(), 'scripts/smoke-orpo-1.5b-local.sh no contiene SMOKE_CLIP'
print(f'[ok] orpo-diag x{n} | smoke con SMOKE_CLIP')


In [ ]:
# CELL 2 — dataset ORPO desde /kaggle/input
import glob, json, os

cand = glob.glob('/kaggle/input/**/*orpo*.jsonl', recursive=True)
print('  candidatos:', cand)
assert cand, 'no encontre el dataset de ORPO en /kaggle/input'
src = cand[0]
rows = [json.loads(l) for l in open(src) if l.strip()]
print(f'  pares: {len(rows)}')
print('  claves:', list(rows[0].keys()))
assert len(rows) == 305, f'esperaba 305 pares, el dataset trae {len(rows)}'
_claves = set(rows[0].keys())
assert _claves == {'prompt', 'chosen', 'rejected'}, f'claves inesperadas: {sorted(_claves)}'
os.makedirs('data', exist_ok=True)
open('data/kateto_orpo_v7.jsonl','w').write(''.join(json.dumps(r,ensure_ascii=False)+'\n' for r in rows))
print('[ok] dataset: 305 pares, claves prompt/chosen/rejected')


In [ ]:
# CELL 3 — ORPO con on_after_backward instrumentado, ctx 256
import subprocess, os, sys

# Base literal: el SFT v7 (kateto-v7-sft). Nada de glob.
BASE_MODEL = '/kaggle/input/kateto-v7-sft/rwkv-0.pth'
assert os.path.exists(BASE_MODEL), f'falta el checkpoint base: {BASE_MODEL}'
_sz = os.path.getsize(BASE_MODEL)
if _sz != 3055569567:
    raise RuntimeError(f'tamaño inesperado de rwkv-0.pth: {_sz} bytes (esperado 3055569567)')

env = dict(os.environ)
env.update({
    'ORPO_DIAG':'1', 'SMOKE_STEPS':'305', 'SMOKE_ACCUM':'4',
    'SMOKE_LR':'5e-7', 'SMOKE_LR_FINAL':'1e-7', 'SMOKE_ORPO_LAMBDA':'0.05',
    'SMOKE_CTX':'256', 'SMOKE_PREC':'bf16', 'SMOKE_GRADCP':'0',
    'SMOKE_MERGE':'1', 'SMOKE_DETACH':'0', 'SMOKE_TRAINTYPE':'none',
    'SMOKE_DATA':'/kaggle/working/kateto/data/kateto_orpo_v7.jsonl',
    'BASE_MODEL': BASE_MODEL,
    'SMOKE_OP':'fla', 'SMOKE_DEVICES':'1', 'SMOKE_STRATEGY':'auto', 'SMOKE_CLIP': '',  # train.py ya fija gradient_clip_val=1.0; no es flag de CLI
    'PYTHON_BIN': sys.executable,
    'REPO_DIR': '/kaggle/working/kateto',
    'SMOKE_OUT':'/kaggle/working/orpo-v7',
})
r = subprocess.run(['bash','scripts/smoke-orpo-1.5b-local.sh'], env=env,
                   capture_output=True, text=True)
log = r.stdout + r.stderr
print(log[-6000:])
print('STDERR:', r.stderr[-3000:])
print('rc =', r.returncode)


### Corrida B — aislamiento fp32: solo si A dio NaN


In [ ]:
# CELL 4 — Corrida B (aislamiento fp32: solo si A dio NaN)
import os, subprocess, sys

r2 = None
log_b = ''

if ('ce=nan' in log) or ('tot_finite=0' in log):
    print('[corrida B] A mostró ce=nan o tot_finite=0; lanzando aislamiento fp32...')
    env_b = dict(os.environ)
    env_b.update({
        'ORPO_DIAG': '1',
        'ORPO_FP32': '1',
        'SMOKE_STEPS': '60',
        'SMOKE_ACCUM': '4',
        'SMOKE_LR': '1e-6',
        'SMOKE_LR_FINAL': '1e-7',
        'SMOKE_ORPO_LAMBDA': '0.05',
        'SMOKE_CTX': '256',
        'SMOKE_PREC': 'bf16',
        'SMOKE_GRADCP': '0',
        'SMOKE_MERGE': '0',
        'SMOKE_DETACH': '0',
        'SMOKE_TRAINTYPE': 'none',
        'SMOKE_DATA': '/kaggle/working/kateto/data/kateto_orpo_v7.jsonl',
        'BASE_MODEL': BASE_MODEL,
        'SMOKE_OP': 'fla',
        'SMOKE_DEVICES': '1',
        'SMOKE_STRATEGY': 'auto',
        'SMOKE_CLIP': '',  # train.py ya fija gradient_clip_val=1.0; no es flag de CLI
        'PYTHON_BIN': sys.executable,
        'REPO_DIR': '/kaggle/working/kateto',
        'SMOKE_OUT': '/kaggle/working/orpo-v7-fp32',
    })
    r2 = subprocess.run(['bash', 'scripts/smoke-orpo-1.5b-local.sh'], env=env_b,
                        capture_output=True, text=True)
    log_b = r2.stdout + r2.stderr
    print('rc =', r2.returncode)
    print('\n'.join(log_b.splitlines()[-60:]))
else:
    print('[skip] Corrida A limpia (sin ce=nan ni tot_finite=0); r2 = None')


In [ ]:
# CELL 5 — veredicto final (cubre A y B)
import re, subprocess, os

lines = []
def P(s=''):
    print(s)
    lines.append(str(s))

# --- Corrida A ---
a_rc = r.returncode
a_fin = len(re.findall(r'\[orpo\] paso \d+ ce=[0-9]', log))
a_nan = len(re.findall(r'\[orpo\] paso \d+ ce=nan', log))
a_diag = re.findall(r'\[orpo-diag [^\]]*\]', log)
a_guard = len(re.findall(r'\[orpo-guard ', log))
a_skip = len(re.findall(r'\[orpo-skip ', log))
a_first_tot0 = None
a_first_lw0 = None
for dl in a_diag:
    m = re.search(r'step=(\d+)', dl)
    st = int(m.group(1)) if m else -1
    if a_first_tot0 is None and 'tot_finite=0' in dl:
        a_first_tot0 = st
    if a_first_lw0 is None and 'lw_finite=0' in dl:
        a_first_lw0 = st

a_had_nan = (a_nan > 0) or (a_first_tot0 is not None) or ('ce=nan' in log) or ('tot_finite=0' in log)

P('=== A (bf16, triton, 305 pasos) ===')
P(f'rc={a_rc} pasos_ce_finito={a_fin} pasos_ce_nan={a_nan} diag={len(a_diag)} guard={a_guard} skip={a_skip}')
P(f'primer_tot_finite_0={a_first_tot0} primer_lw_finite_0={a_first_lw0}')

# --- Corrida B (si disponible) ---
b_had_nan = False
if r2 is not None:
    b_rc = r2.returncode
    b_fin = len(re.findall(r'\[orpo\] paso \d+ ce=[0-9]', log_b))
    b_nan = len(re.findall(r'\[orpo\] paso \d+ ce=nan', log_b))
    b_diag = re.findall(r'\[orpo-diag [^\]]*\]', log_b)
    b_first_tot0 = None
    b_first_lw0 = None
    for dl in b_diag:
        m = re.search(r'step=(\d+)', dl)
        st = int(m.group(1)) if m else -1
        if b_first_tot0 is None and 'tot_finite=0' in dl:
            b_first_tot0 = st
        if b_first_lw0 is None and 'lw_finite=0' in dl:
            b_first_lw0 = st
    b_had_nan = (b_nan > 0) or (b_first_tot0 is not None) or ('ce=nan' in log_b) or ('tot_finite=0' in log_b)

    P('=== B (fp32 real, 60 pasos) ===')
    P(f'rc={b_rc} pasos_ce_finito={b_fin} pasos_ce_nan={b_nan} primer_tot_finite_0={b_first_tot0} primer_lw_finite_0={b_first_lw0}')

# --- Evidencia forward lw_finite=0 en A ---
for l in log.splitlines():
    if '[orpo-diag step=' in l and 'lw_finite=0' in l:
        P(l)
        break

# --- Veredicto final ---
if a_rc != 0:
    veredicto = f'VEREDICTO: NO CORRIÓ EN A (rc={a_rc})'
elif a_nan == 0 and not a_had_nan and a_fin > 0:
    veredicto = 'VEREDICTO: LIMPIO EN BF16'
elif a_had_nan:
    if r2 is not None and b_rc == 0 and b_nan == 0 and not b_had_nan and b_fin > 0:
        veredicto = 'VEREDICTO: BF16 NA el NaN, FP32 LIMPIO'
    else:
        veredicto = 'VEREDICTO: NaN TAMBIEN EN T4, causa pendiente'
else:
    veredicto = f'VEREDICTO: NO CORRIÓ EN A (rc={a_rc})'

P(veredicto)

# --- Últimas 120 líneas del log de A si A tuvo NaN o no corrió ---
if a_rc != 0 or a_had_nan or a_fin == 0:
    P('--- últimas 120 líneas del log de A ---')
    for _l in log.splitlines()[-120:]:
        P(_l)

os.makedirs('/kaggle/working', exist_ok=True)
with open('/kaggle/working/veredicto.txt', 'w') as f:
    f.write('\n'.join(lines) + '\n')
print('[ok] veredicto escrito en /kaggle/working/veredicto.txt')
